dataset: data/nmt/demo.txt
For example below, One line one sample in the format: english sentence. \t chinese translation.
She put the magazine on the table.	她把雜誌放在桌上。
Hey, what are you doing here?	嘿，你在這做什麼？
Please keep this secret.	請保守這個秘密。
How could things get worse?	事情怎麼變糟的？
Kyoto and Boston are sister cities.	京都和波士顿是姐妹城市。
Tom mostly kept to himself.	汤姆大多一个人独处。
Do you like music?	你爱音乐吗？
Tell me the reason why she got angry.	告诉我她为什么生气。

In [100]:
from nltk import word_tokenize
import os
from collections import Counter
import numpy as np

print(f"Current directory: {os.getcwd()}")

Current directory: /home/zds/trunk/robot/lerobot/py101/src/py101/transformer/tutorial


In [ ]:
#
# Load the tokens(sentence) list
#
def load_dataset(path):
    en_tokens_list, cn_tokens_list = [], []
    index = 0
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            index += 1
            sample = line.strip().split('\t')
            if len(sample) == 2:
                print(f"[{index}]:{sample[0]} -> {sample[1]}")
                # english sentence tokens
                en_tokens = word_tokenize(sample[0].lower())
                print(f"\ten_tokens: {type(en_tokens)} len:{len(en_tokens)} content:{en_tokens}")
                cn_tokens = word_tokenize(" ".join([w for w in sample[1]]))
                print(f"\tcn_tokens: {type(cn_tokens)} len:{len(cn_tokens)} content:{cn_tokens}")
                en_tokens_list.append(['BOS'] + en_tokens + ['EOS'])
                cn_tokens_list.append(['BOS'] + cn_tokens + ['EOS'])
    return en_tokens_list, cn_tokens_list

In [50]:
demo_file = "demo.txt"

en, cn = load_dataset(demo_file)
for i in range(len(en)):
    print(f"[{i+1}]: {en[i]} -> {cn[i]}")

[1]:She put the magazine on the table. -> 她把雜誌放在桌上。
	en_tokens: <class 'list'> len:8 content:['she', 'put', 'the', 'magazine', 'on', 'the', 'table', '.']
	cn_tokens: <class 'list'> len:9 content:['她', '把', '雜', '誌', '放', '在', '桌', '上', '。']
[2]:Hey, what are you doing here? -> 嘿，你在這做什麼？
	en_tokens: <class 'list'> len:8 content:['hey', ',', 'what', 'are', 'you', 'doing', 'here', '?']
	cn_tokens: <class 'list'> len:9 content:['嘿', '，', '你', '在', '這', '做', '什', '麼', '？']
[3]:Please keep this secret. -> 請保守這個秘密。
	en_tokens: <class 'list'> len:5 content:['please', 'keep', 'this', 'secret', '.']
	cn_tokens: <class 'list'> len:8 content:['請', '保', '守', '這', '個', '秘', '密', '。']
[4]:How could things get worse? -> 事情怎麼變糟的？
	en_tokens: <class 'list'> len:6 content:['how', 'could', 'things', 'get', 'worse', '?']
	cn_tokens: <class 'list'> len:8 content:['事', '情', '怎', '麼', '變', '糟', '的', '？']
[5]:Kyoto and Boston are sister cities. -> 京都和波士顿是姐妹城市。
	en_tokens: <class 'list'> len:7 content:['kyoto',

In [ ]:
#
# build the vocabulary from tokens(sentence) list
#
def build_vocab(tokens_list, max_vocab_size=100):
    vocab = {}
    token_count = Counter()
    # counting the tokens
    for tokens in tokens_list:
        for token in tokens:
            token_count[token] += 1
    # get the most common token list
    most_common_tokens = token_count.most_common(max_vocab_size)
    print(f"most_common_tokens type:{type(most_common_tokens)} len:{len(most_common_tokens)} the first 5:\n\t{most_common_tokens[:5]}")
    
    vocab_size = len(most_common_tokens) + 2 # add 2 more for 'UNK'(index=0) and 'PAD'(index=1)
    print(f"vocab_size: {vocab_size}")
    # sort the vocab by common frequency in the dataset {token: idx}
    vocab = {token: idx + 2 for idx, (token, count) in enumerate(most_common_tokens)}
    print(f"vocab type:{type(vocab)}: the first 5\n\t{list(vocab.items())[:5]}")
    # inverted index: {idx: token}
    index_vocab = {idx: token for token, idx in vocab.items()}
    print(f"index vocab type:{type(index_vocab)}: the first 5\n\t{list(index_vocab.items())[:5]}")
    
    return vocab, vocab_size, index_vocab

In [64]:
en_vocab, en_vocab_size, en_index_vocab = build_vocab(en)
cn_vocab, cn_vocab_size, cn_index_vocab = build_vocab(cn)

most_common_tokens type:<class 'list'> len:45 the first 5:
	[('BOS', 8), ('EOS', 8), ('.', 5), ('the', 3), ('?', 3)]
vocab_size: 47
vocab type:<class 'dict'>: the first 5
	[('BOS', 2), ('EOS', 3), ('.', 4), ('the', 5), ('?', 6)]
index vocab type:<class 'dict'>: the first 5
	[(2, 'BOS'), (3, 'EOS'), (4, '.'), (5, 'the'), (6, '?')]
most_common_tokens type:<class 'list'> len:62 the first 5:
	[('BOS', 8), ('EOS', 8), ('。', 5), ('？', 3), ('她', 2)]
vocab_size: 64
vocab type:<class 'dict'>: the first 5
	[('BOS', 2), ('EOS', 3), ('。', 4), ('？', 5), ('她', 6)]
index vocab type:<class 'dict'>: the first 5
	[(2, 'BOS'), (3, 'EOS'), (4, '。'), (5, '？'), (6, '她')]


In [78]:
def len_argsort(seq):
    """
    get sorted index w.r.t length.
    Returns the indices that would sort a sequence by the length of its elements (from shortest to longest).
    """
    return sorted(range(len(seq)), key=lambda x: len(seq[x]))       

In [66]:
sorted_en_tokens_indices = len_argsort(en)
for i in range(len(sorted_en_tokens_indices)):
    print(f"[{i+1}]: {sorted_en_tokens_indices[i]} len:{len(en[sorted_en_tokens_indices[i]])} {en[sorted_en_tokens_indices[i]]}")

[1]: 2 len:7 ['BOS', 'please', 'keep', 'this', 'secret', '.', 'EOS']
[2]: 6 len:7 ['BOS', 'do', 'you', 'like', 'music', '?', 'EOS']
[3]: 3 len:8 ['BOS', 'how', 'could', 'things', 'get', 'worse', '?', 'EOS']
[4]: 5 len:8 ['BOS', 'tom', 'mostly', 'kept', 'to', 'himself', '.', 'EOS']
[5]: 4 len:9 ['BOS', 'kyoto', 'and', 'boston', 'are', 'sister', 'cities', '.', 'EOS']
[6]: 0 len:10 ['BOS', 'she', 'put', 'the', 'magazine', 'on', 'the', 'table', '.', 'EOS']
[7]: 1 len:10 ['BOS', 'hey', ',', 'what', 'are', 'you', 'doing', 'here', '?', 'EOS']
[8]: 7 len:11 ['BOS', 'tell', 'me', 'the', 'reason', 'why', 'she', 'got', 'angry', '.', 'EOS']


In [96]:
#
# tokens(sentence) list to tensor(token index in the vocabulary) list
#
def tokensToIds(en_tokens_list, cn_tokens_list, en_vocab, cn_vocab, sort=True):
    UNK_INDEX = 0
    en_embed_list = [[en_vocab.get(token, UNK_INDEX) for token in tokens] for tokens in en_tokens_list]
    cn_tokens_list = [[cn_vocab.get(token, UNK_INDEX) for token in tokens] for tokens in cn_tokens_list]

    if sort:  # update index
        sorted_index = len_argsort(en_tokens_list)  # English
        print(f"sorted index:{sorted_index}")
        en_embed_list = [en_embed_list[id] for id in sorted_index]
        cn_tokens_list = [cn_tokens_list[id] for id in sorted_index]
            
    return en_embed_list, cn_tokens_list

In [89]:
en_embed, cn_embed = tokensToIds(en, cn, en_vocab, cn_vocab, False)
for i in range(len(en_embed)):
    print(f"English: {[en_index_vocab[id] for id in en_embed[i]]}")
    print(f"Chinese: {[cn_index_vocab[id] for id in cn_embed[i]]}")
    print(f"English embedding: {en_embed[i]}")
    print(f"Chinese embedding: {cn_embed[i]}")
    print("-" * 50)

English: ['BOS', 'she', 'put', 'the', 'magazine', 'on', 'the', 'table', '.', 'EOS']
Chinese: ['BOS', '她', '把', '雜', '誌', '放', '在', '桌', '上', '。', 'EOS']
English embedding: [2, 7, 10, 5, 11, 12, 5, 13, 4, 3]
Chinese embedding: [2, 6, 12, 13, 14, 15, 7, 16, 17, 4, 3]
--------------------------------------------------
English: ['BOS', 'hey', ',', 'what', 'are', 'you', 'doing', 'here', '?', 'EOS']
Chinese: ['BOS', '嘿', '，', '你', '在', '這', '做', '什', '麼', '？', 'EOS']
English embedding: [2, 14, 15, 16, 8, 9, 17, 18, 6, 3]
Chinese embedding: [2, 18, 19, 8, 7, 9, 20, 10, 11, 5, 3]
--------------------------------------------------
English: ['BOS', 'please', 'keep', 'this', 'secret', '.', 'EOS']
Chinese: ['BOS', '請', '保', '守', '這', '個', '秘', '密', '。', 'EOS']
English embedding: [2, 19, 20, 21, 22, 4, 3]
Chinese embedding: [2, 21, 22, 23, 9, 24, 25, 26, 4, 3]
--------------------------------------------------
English: ['BOS', 'how', 'could', 'things', 'get', 'worse', '?', 'EOS']
Chinese: ['BOS', '

In [97]:
sorted_en_embed, sorted_cn_embed = tokensToIds(en, cn, en_vocab, cn_vocab, True)
for i in range(len(sorted_en_embed)):
    print(f"English: {[en_index_vocab[id] for id in sorted_en_embed[i]]}")
    print(f"Chinese: {[cn_index_vocab[id] for id in sorted_cn_embed[i]]}")
    print(f"English embedding: {sorted_en_embed[i]}")
    print(f"Chinese embedding: {sorted_cn_embed[i]}")
    print("-" * 50)

sorted index:[2, 6, 3, 5, 4, 0, 1, 7]
English: ['BOS', 'please', 'keep', 'this', 'secret', '.', 'EOS']
Chinese: ['BOS', '請', '保', '守', '這', '個', '秘', '密', '。', 'EOS']
English embedding: [2, 19, 20, 21, 22, 4, 3]
Chinese embedding: [2, 21, 22, 23, 9, 24, 25, 26, 4, 3]
--------------------------------------------------
English: ['BOS', 'do', 'you', 'like', 'music', '?', 'EOS']
Chinese: ['BOS', '你', '爱', '音', '乐', '吗', '？', 'EOS']
English embedding: [2, 38, 9, 39, 40, 6, 3]
Chinese embedding: [2, 8, 53, 54, 55, 56, 5, 3]
--------------------------------------------------
English: ['BOS', 'how', 'could', 'things', 'get', 'worse', '?', 'EOS']
Chinese: ['BOS', '事', '情', '怎', '麼', '變', '糟', '的', '？', 'EOS']
English embedding: [2, 23, 24, 25, 26, 27, 6, 3]
Chinese embedding: [2, 27, 28, 29, 11, 30, 31, 32, 5, 3]
--------------------------------------------------
English: ['BOS', 'tom', 'mostly', 'kept', 'to', 'himself', '.', 'EOS']
Chinese: ['BOS', '汤', '姆', '大', '多', '一', '个', '人', '独', '处', 

In [ ]:
#
# split into batches
def seq_padding(X, padding=0):
    """
    add padding to a batch data
    """
    L = [len(x) for x in X]
    ML = max(L)
    return np.array([
        np.concatenate([x, [padding] * (ML - len(x))]) if len(x) < ML else x for x in X
    ])

def splitBatch(en_embed, cn_embed, batch_size, shuffle=True):
    idx_list = np.arange(0, len(en_embed), batch_size)
    print(f"Total {len(idx_list)} batches with batch size {batch_size}, batch start index: {idx_list}")

    if shuffle:
        np.random.shuffle(idx_list)
        print(f"shuffed batch start index: {idx_list}")
    
    batch_indices = []
    for i, start_index in enumerate(idx_list):
        batch_index = np.arange(start_index, min(start_index + batch_size, len(en_embed)))
        batch_indices.append(batch_index)
        print(f"B[{i}]:{batch_index}")
    
    en_batches, cn_batches = [], []
    for i, batch_index in enumerate(batch_indices):
        en_batch = [en_embed[idx] for idx in batch_index]
        cn_batch = [cn_embed[idx] for idx in batch_index]
        print(f"B[{i}]:\n\ten_batch:\n\t\t{en_batch}\n\tcn_batch:\n\t\t{cn_batch}")
        
        en_batch = seq_padding(en_batch)
        cn_batch = seq_padding(cn_batch)
        print(f"\n\tpad en_batch:\n\t\t{en_batch.tolist()}\n\tpad cn_batch:\n\t\t{cn_batch.tolist()}")
        en_batches.append(en_batch)
        cn_batches.append(cn_batch)
    
    return en_batches, cn_batches

In [137]:
batch_size = 2
en_batches, cn_batches = splitBatch(sorted_en_embed, sorted_cn_embed, batch_size, True)

Total 4 batches with batch size 2, batch start index: [0 2 4 6]
shuffed batch start index: [2 0 4 6]
B[0]:[2 3]
B[1]:[0 1]
B[2]:[4 5]
B[3]:[6 7]
B[0]:
	en_batch:
		[[2, 23, 24, 25, 26, 27, 6, 3], [2, 33, 34, 35, 36, 37, 4, 3]]
	cn_batch:
		[[2, 27, 28, 29, 11, 30, 31, 32, 5, 3], [2, 44, 45, 46, 47, 48, 49, 50, 51, 52, 4, 3]]

	pad en_batch:
		[[2, 23, 24, 25, 26, 27, 6, 3], [2, 33, 34, 35, 36, 37, 4, 3]]
	pad cn_batch:
		[[2, 27, 28, 29, 11, 30, 31, 32, 5, 3, 0, 0], [2, 44, 45, 46, 47, 48, 49, 50, 51, 52, 4, 3]]
B[1]:
	en_batch:
		[[2, 19, 20, 21, 22, 4, 3], [2, 38, 9, 39, 40, 6, 3]]
	cn_batch:
		[[2, 21, 22, 23, 9, 24, 25, 26, 4, 3], [2, 8, 53, 54, 55, 56, 5, 3]]

	pad en_batch:
		[[2, 19, 20, 21, 22, 4, 3], [2, 38, 9, 39, 40, 6, 3]]
	pad cn_batch:
		[[2, 21, 22, 23, 9, 24, 25, 26, 4, 3], [2, 8, 53, 54, 55, 56, 5, 3, 0, 0]]
B[2]:
	en_batch:
		[[2, 28, 29, 30, 8, 31, 32, 4, 3], [2, 7, 10, 5, 11, 12, 5, 13, 4, 3]]
	cn_batch:
		[[2, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 4, 3], [2, 